# 00 – Environment Check

**Purpose:** Validate that the SupportAI development environment is correctly configured.

This notebook:
- Confirms Python version ≥ 3.12
- Imports all core libraries and prints versions
- Confirms CPU is available (and reports GPU if present)
- Verifies project path constants resolve correctly
- Confirms logging and seeding utilities work end-to-end
- Detects whether we are running on Kaggle or locally

**Run this notebook after every environment change (new machine, `pip install`, etc.).**

---
> ⚠️ This notebook contains NO production logic. It only calls `src/` functions.

## 0. Setup – Add repo root to sys.path

Needed when running outside of an installed editable package.

In [1]:
import sys
from pathlib import Path

# Resolve repo root (notebooks/ is one level below root)
REPO_ROOT = Path().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root : {REPO_ROOT}")
print(f"sys.path[0]: {sys.path[0]}")

Repo root : C:\Users\gunav\Downloads\SupportAI
sys.path[0]: C:\Users\gunav\Downloads\SupportAI


## 1. Python & Platform

In [2]:
import platform
import sys

print("=" * 55)
print("  Python & Platform")
print("=" * 55)
print(f"Python version  : {sys.version}")
print(f"Python executable: {sys.executable}")
print(f"Platform        : {platform.platform()}")
print(f"Architecture    : {platform.machine()}")

major, minor = sys.version_info.major, sys.version_info.minor
assert (major, minor) >= (3, 12), f"⛔  Python 3.12+ required. Got {major}.{minor}"
print(f"\n✅  Python {major}.{minor} – OK")

  Python & Platform
Python version  : 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
Python executable: c:\Users\gunav\anaconda3\python.exe
Platform        : Windows-10-10.0.19045-SP0
Architecture    : AMD64

✅  Python 3.13 – OK


## 2. Kaggle / Local Environment Detection

In [3]:
import os

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").exists()

print("=" * 55)
print("  Execution Environment")
print("=" * 55)
print(f"Running on Kaggle : {IS_KAGGLE}")
print(f"Working directory  : {Path.cwd()}")

if IS_KAGGLE:
    print("📓  Kaggle environment detected – GPU may be available.")
else:
    print("💻  Local environment detected – CPU-only mode expected.")

  Execution Environment
Running on Kaggle : True
Working directory  : c:\Users\gunav\Downloads\SupportAI
📓  Kaggle environment detected – GPU may be available.


## 3. Core Library Versions

In [4]:
import importlib

PACKAGES = [
    "torch",
    "transformers",
    "datasets",
    "sklearn",
    "pandas",
    "numpy",
    "mlflow",
    "matplotlib",
    "seaborn",
    "tqdm",
    "yaml",
    "rich",
    "scipy",
]

print("=" * 55)
print("  Library Versions")
print("=" * 55)

missing = []
for pkg in PACKAGES:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "<no __version__>")
        print(f"  {'✅':<4} {pkg:<20} {version}")
    except ImportError:
        print(f"  {'❌':<4} {pkg:<20} NOT FOUND")
        missing.append(pkg)

print()
if missing:
    print(f"⚠️  Missing packages: {missing}")
    print("   Run: pip install -e '.[dev]'")
else:
    print("✅  All packages imported successfully.")

  Library Versions
  ✅    torch                2.10.0+cpu
  ✅    transformers         5.13.1
  ✅    datasets             4.1.1
  ✅    sklearn              1.6.1
  ✅    pandas               2.2.3
  ✅    numpy                2.1.3
  ✅    mlflow               3.14.0
  ✅    matplotlib           3.10.0
  ✅    seaborn              0.13.2
  ✅    tqdm                 4.67.1
  ✅    yaml                 6.0.2
  ✅    rich                 <no __version__>
  ✅    scipy                1.15.3

✅  All packages imported successfully.


## 4. Hardware / Device Check

In [5]:
import torch

print("=" * 55)
print("  Hardware / Device Check")
print("=" * 55)
print(f"PyTorch version    : {torch.__version__}")
print(f"CUDA available     : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version       : {torch.version.cuda}")
    print(f"GPU count          : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  ({props.total_memory / 1e9:.1f} GB VRAM)")
    device = "cuda"
else:
    import os
    cpu_count = os.cpu_count()
    print(f"CPU logical cores  : {cpu_count}")
    print("MPS available      :", getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available())
    device = "cpu"

print(f"\nActive device      : {device}")

# Quick tensor round-trip test
t = torch.tensor([1.0, 2.0, 3.0]).to(device)
assert t.sum().item() == 6.0, "Tensor computation sanity check failed!"
print("✅  Tensor computation OK.")

  Hardware / Device Check
PyTorch version    : 2.10.0+cpu
CUDA available     : False
CPU logical cores  : 8
MPS available      : False

Active device      : cpu
✅  Tensor computation OK.


## 5. Project Path Constants

In [6]:
from src.utils.constants import (
    BASE_DIR,
    CHECKPOINTS_DIR,
    CONFIG_DIR,
    DATA_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    METRICS_DIR,
    MLFLOW_TRACKING_URI,
    MODELS_DIR,
    OUTPUT_DIR,
)

print("=" * 55)
print("  Project Path Constants")
print("=" * 55)

path_checks = [
    ("BASE_DIR",         BASE_DIR,         True),
    ("CONFIG_DIR",       CONFIG_DIR,       True),
    ("OUTPUT_DIR",       OUTPUT_DIR,       False),   # may not exist yet
    ("DATA_DIR",         DATA_DIR,         False),
    ("MODELS_DIR",       MODELS_DIR,       False),
    ("METRICS_DIR",      METRICS_DIR,      False),
    ("FIGURES_DIR",      FIGURES_DIR,      False),
    ("CHECKPOINTS_DIR",  CHECKPOINTS_DIR,  False),
    ("LOGS_DIR",         LOGS_DIR,         False),
]

for name, path, must_exist in path_checks:
    status = "✅" if (not must_exist or path.exists()) else "⚠️ "
    exists = "exists" if path.exists() else "not yet created"
    print(f"  {status}  {name:<20} {path}  [{exists}]")

print(f"\n  MLflow URI  : {MLFLOW_TRACKING_URI}")
print("\n✅  Path constants resolved correctly.")

  Project Path Constants
  ✅  BASE_DIR             C:\Users\gunav\Downloads\SupportAI  [exists]
  ✅  CONFIG_DIR           C:\Users\gunav\Downloads\SupportAI\configs  [exists]
  ✅  OUTPUT_DIR           C:\Users\gunav\Downloads\SupportAI\outputs  [exists]
  ✅  DATA_DIR             C:\Users\gunav\Downloads\SupportAI\data  [exists]
  ✅  MODELS_DIR           C:\Users\gunav\Downloads\SupportAI\outputs\models  [exists]
  ✅  METRICS_DIR          C:\Users\gunav\Downloads\SupportAI\outputs\metrics  [exists]
  ✅  FIGURES_DIR          C:\Users\gunav\Downloads\SupportAI\outputs\figures  [exists]
  ✅  CHECKPOINTS_DIR      C:\Users\gunav\Downloads\SupportAI\outputs\checkpoints  [exists]
  ✅  LOGS_DIR             C:\Users\gunav\Downloads\SupportAI\logs  [exists]

  MLflow URI  : file:///C:/Users/gunav/Downloads/SupportAI/outputs/mlruns

✅  Path constants resolved correctly.


## 6. Logging Utility

In [7]:
from src.utils.logging_utils import get_logger

logger = get_logger("notebook.env_check")
logger.info("Logging utility: ✅ working")
logger.debug("Debug messages are visible when level=DEBUG")
logger.warning("This is a sample warning – not a real problem.")
print("\n✅  Logging OK – check output above for formatted log lines.")

[07/13/26 09:12:42] INFO     Logging utility: ✅ working

                    WARNING  This is a sample warning – not a real problem.


✅  Logging OK – check output above for formatted log lines.


## 7. Seed Utility

In [8]:
import random

import numpy as np
import torch
from src.utils.seed import DEFAULT_SEED, set_seed

set_seed(DEFAULT_SEED)
r1 = random.random()
n1 = np.random.rand()
t1 = torch.rand(1).item()

set_seed(DEFAULT_SEED)
r2 = random.random()
n2 = np.random.rand()
t2 = torch.rand(1).item()

assert r1 == r2, "random reproducibility failed"
assert n1 == n2, "numpy reproducibility failed"
assert t1 == t2, "torch reproducibility failed"

print("=" * 55)
print("  Seed Reproducibility Check")
print("=" * 55)
print(f"  Default seed   : {DEFAULT_SEED}")
print(f"  random.random(): {r1:.6f}  (matches after re-seed)")
print(f"  np.random.rand(): {n1:.6f}  (matches after re-seed)")
print(f"  torch.rand(1)  : {t1:.6f}  (matches after re-seed)")
print("\n✅  Seed utility: reproducibility confirmed.")

  Seed Reproducibility Check
  Default seed   : 42
  random.random(): 0.639427  (matches after re-seed)
  np.random.rand(): 0.374540  (matches after re-seed)
  torch.rand(1)  : 0.882269  (matches after re-seed)

✅  Seed utility: reproducibility confirmed.


## 8. Summary

Print a final pass/fail summary for the environment check.

In [9]:
from src.utils.constants import BASE_DIR

print("=" * 55)
print("  SupportAI Environment Check – Summary")
print("=" * 55)
summary = [
    ("Python 3.12+",             True),
    ("All packages importable",  not missing if 'missing' in dir() else True),
    ("Tensor computation",       True),
    ("Path constants resolved",  True),
    ("Logging utility",          True),
    ("Seed reproducibility",     True),
]

all_pass = True
for label, ok in summary:
    icon = "✅" if ok else "❌"
    print(f"  {icon}  {label}")
    if not ok:
        all_pass = False

print()
if all_pass:
    print("🎉  All checks passed. Environment is ready for Phase 2.")
else:
    print("⚠️  Some checks failed. Review output above and fix before proceeding.")

  SupportAI Environment Check – Summary
  ✅  Python 3.12+
  ✅  All packages importable
  ✅  Tensor computation
  ✅  Path constants resolved
  ✅  Logging utility
  ✅  Seed reproducibility

🎉  All checks passed. Environment is ready for Phase 2.


In [10]:
# Export Phase Manifest
import sys

import torch
from src.api.app import get_git_commit
from src.utils.artifacts import save_yaml

manifest = {
    "phase": "00_Environment_Check",
    "python_version": sys.version,
    "pytorch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "git_commit": get_git_commit(),
}
save_yaml(manifest, REPO_ROOT / "outputs" / "manifests" / "phase_00_environment_check.yaml")
print("YAML manifest saved successfully:")
print(manifest)


[07/13/26 09:12:49] INFO     Loading faiss with AVX2 support.

                    INFO     Could not load library with AVX2 support due to:                                      
                             ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")

                    INFO     Loading faiss.

                    INFO     Successfully loaded faiss.

[07/13/26 09:12:50] INFO     Saving YAML artifact to:                                                              
                             C:\Users\gunav\Downloads\SupportAI\outputs\manifests\phase_00_environment_check.yaml

YAML manifest saved successfully:
{'phase': '00_Environment_Check', 'python_version': '3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]', 'pytorch_version': '2.10.0+cpu', 'cuda_available': False, 'git_commit': 'cc1c1de64b28542d4db316ac79bd4990d31964a5'}
